# 서버 원본 SMILES 매핑 및 Morgan 생성 근거

`디퓨전 2차 시험 (1).ipynb` 107번째 셀의 저장 출력에서 추출했습니다. 원본 코드와 수동 값은 보존하며 이 환경에서 실행하지 않았습니다. `compounds_137`의 구성은 제공되지 않았으므로 준비해야 합니다. 아래 PubChem 요청과 CSV 쓰기는 실행 시 발생합니다. 첫 검색 결과 및 수동 CID·SMILES의 물질 동일성 검증은 별도입니다.

기존 legacy 생성 스크립트는 `scripts/original_server/make_structure_features.py`, 최종 matched 입력에 대응하는 서버 스크립트는 `scripts/structure_features.py`로 보존했습니다. 둘은 fragment 처리·보간·열 제거·fingerprint 생성 방식이 다릅니다. 저장 config는 `results/morgan_matched_config.observed.json`을 참고하세요. 이 노트북에는 학습 코드가 없습니다.


## Untitled.ipynb 원본 코드 인덱스 27


In [ ]:
import pubchempy as pcp

test_compounds = [
    "acetaminophen",
    "caffeine",
    "ibuprofen",
    "valproic acid",
    "WY-14643"
]

for name in test_compounds:
    try:
        results = pcp.get_compounds(name, "name")

        if len(results) == 0:
            print(f"❌ {name}: NOT FOUND")
            continue

        c = results[0]

        print(f"✅ {name}")
        print("   CID   :", c.cid)
        print("   SMILES:", c.connectivity_smiles)
        print()

    except Exception as e:
        print(f"⚠️ {name}: ERROR")
        print("  ", e)


## Untitled.ipynb 원본 코드 인덱스 28


In [ ]:
import pubchempy as pcp
import pandas as pd
import time

smiles_results = []

for i, name in enumerate(compounds_137, 1):

    print(f"[{i:3d}/{len(compounds_137)}] {name}", end=" ... ")

    success = False
    last_error = None

    # 최대 3번 재시도
    for attempt in range(3):

        try:
            compounds = pcp.get_compounds(name, "name")

            if len(compounds) > 0:
                c = compounds[0]

                smiles_results.append({
                    "compound": name,
                    "pubchem_cid": c.cid,
                    "smiles": c.connectivity_smiles,
                    "status": "OK"
                })

                print(f"OK (CID {c.cid})")
                success = True
                break

            else:
                last_error = "NOT FOUND"
                break

        except Exception as e:
            last_error = str(e)

            # 일시적인 PubChem 오류일 수 있으므로 잠깐 대기
            time.sleep(2)


    if not success:

        smiles_results.append({
            "compound": name,
            "pubchem_cid": None,
            "smiles": None,
            "status": last_error
        })

        print("FAILED")


    # PubChem에 너무 빠르게 요청하지 않도록 대기
    time.sleep(0.3)


# DataFrame 생성
smiles_df = pd.DataFrame(smiles_results)


# CSV 저장
smiles_df.to_csv(
    "compound_pubchem_smiles.csv",
    index=False
)


print("\n==============================")
print("완료")
print("==============================")

print("전체:", len(smiles_df))
print("성공:", (smiles_df["status"] == "OK").sum())
print("실패:", (smiles_df["status"] != "OK").sum())


print("\n=== 실패한 화합물 ===")

display(
    smiles_df[
        smiles_df["status"] != "OK"
    ]
)


## Untitled.ipynb 원본 코드 인덱스 29


In [ ]:
# acetamidofluorene
mask1 = smiles_df["compound"] == "acetamidofluorene"

smiles_df.loc[mask1, "pubchem_cid"] = 5897
smiles_df.loc[mask1, "smiles"] = "CC(=O)NC1=CC2=C(C=C1)C3=CC=CC=C3C2"
smiles_df.loc[mask1, "status"] = "OK_MANUAL"


# naphthyl isothiocyanate
mask2 = smiles_df["compound"] == "naphthyl isothiocyanate"

smiles_df.loc[mask2, "pubchem_cid"] = 11080
smiles_df.loc[mask2, "smiles"] = "C1=CC=C2C(=C1)C=CC=C2N=C=S"
smiles_df.loc[mask2, "status"] = "OK_MANUAL"


# 다시 저장
smiles_df.to_csv(
    "compound_pubchem_smiles.csv",
    index=False
)


print("전체:", len(smiles_df))
print("SMILES 있음:", smiles_df["smiles"].notna().sum())
print("SMILES 없음:", smiles_df["smiles"].isna().sum())

display(
    smiles_df[
        smiles_df["compound"].isin([
            "acetamidofluorene",
            "naphthyl isothiocyanate"
        ])
    ]
)
